# Bronze CDC Ingestion

## Import Helper Functions

In [1]:
from src import *

# while it is allowed to use the import above, it is advised to list out what we have imported
# from src.config_loader import load_config
# from src.spark_sql_magic import sql
# from src.avro_utils import fetch_avro_schema

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


## Load Configs

In [2]:
cfg = load_config()

JOB_NAMES = cfg["spark_jobs"]["jobs"]
CATALOG = cfg["general"]["catalog"]
BRONZE_NAMESPACE = cfg["general"]["namespaces"]["bronze"]
CHECKPOINT_BASE = cfg["kafka"]["checkpoint_base"]
KAFKA_BOOTSTRAP = cfg["kafka"]["bootstrap_servers"]
TOPIC_PREFIX = cfg["kafka"]["topic_prefix"]
TOPIC_SUFFIX = set(cfg["kafka"]["topic_suffix"])
APICURIO_URL = cfg["api"]["apicurio_url"]

## Import Libraries and Start Session

In [3]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import Window as W
import pyspark
import datetime
import json

spark = (
    SparkSession.builder
        .appName(JOB_NAMES["bronze_cdc_ingestion_backfill"])
        .getOrCreate()
)

## Read Kafka Stream for all topics

In [4]:
kafka_df = (
    spark.readStream
        .format("kafka")
        .option("kafka.bootstrap.servers", KAFKA_BOOTSTRAP)
        .option("subscribePattern", f"{TOPIC_PREFIX}\\..*")    # <- dynamic subscription
        .option("startingOffsets", "latest")
        .load()
)

## Rename Columns and add timestamp

In [5]:
kafka_df = (
    kafka_df
    .withColumn("raw_value", F.col("value"))
    .withColumn("spark_ingest_ts", F.current_timestamp())
    .withColumnRenamed("offset", "kafka_offset")
    .withColumnRenamed("partition", "kafka_partition")
    .withColumnRenamed("timestamp", "kafka_ingest_ts")
)

## Function: foreachBatch writer

In [6]:
from pyspark.sql.avro.functions import from_avro

def write_batch(batch_df, batch_id):
    if batch_df.isEmpty():
        return

    batch_df = (
        batch_df.withColumn("batch_id", F.lit(str(batch_id)))
        .withColumn("topic_suffix", F.regexp_extract(F.col("topic"), r"[^.]+$", 0))
    ).persist()

    # Process per topic dynamically
    topics = [r.topic_suffix for r in batch_df.select("topic_suffix").distinct().collect()]

    for topic in topics:
        topic_df = batch_df.filter(F.col("topic_suffix") == topic)
        table_name = f"{CATALOG}.{BRONZE_NAMESPACE}.{topic}"
        dlq_table = f"{CATALOG}.{BRONZE_NAMESPACE}_dlq.{topic}_dlq"
        audit_table = f"{CATALOG}.{BRONZE_NAMESPACE}_audit.{topic}_audit"

        schema_dict = fetch_avro_schema(topic, APICURIO_URL, TOPIC_PREFIX)
        schema_json = json.dumps(schema_dict)

        decoded_df = topic_df.select(
            from_avro("raw_value", schema_json, {"mode": "PERMISSIVE"}).alias("data"),
            "raw_value",
            "kafka_offset",
            "kafka_partition",
            "kafka_ingest_ts",
            "spark_ingest_ts",
            "batch_id"
        )

        valid_df = decoded_df.filter(F.col("data").isNotNull())
        dlq_df   = decoded_df.filter(F.col("data").isNull())

        valid_df = (
            valid_df
            .select("data.*", "kafka_offset", "kafka_partition", "kafka_ingest_ts", "spark_ingest_ts", "batch_id")
        )

        dlq_df = (
            dlq_df
            .select("raw_value", "kafka_offset", "kafka_partition", "kafka_ingest_ts", "spark_ingest_ts", "batch_id")
        )

        # --- Write bronze and DLQ ---
        valid_df.write.format("iceberg").mode("append").option("mergeSchema", "true").saveAsTable(table_name)
        dlq_df.write.format("iceberg").mode("append").saveAsTable(dlq_table)

        # --- Audit ---
        audit_partition_df = topic_df.groupBy("kafka_partition").agg(
            F.lit(f"{TOPIC_PREFIX}.{topic}").alias("topic"),
            F.lit(str(batch_id)).alias("batch_id"),
            F.current_timestamp().alias("batch_ts"),
            F.min("kafka_offset").alias("min_offset"),
            F.max("kafka_offset").alias("max_offset"),
            F.count("*").alias("row_count")
        )
        audit_partition_df.write.format("iceberg").mode("append").saveAsTable(audit_table)

    # Clean up persistence
    batch_df.unpersist()

## Start Streams

In [7]:
query = (
    kafka_df.writeStream
        .foreachBatch(write_batch)
        .option("checkpointLocation", f"{CHECKPOINT_BASE}/{BRONZE_NAMESPACE}/all_topics")
        .trigger(availableNow=True)  # Databricks-friendly
        .start()
)

query.awaitTermination()

## Sanity Check Show All Tables

In [8]:
%%sql
SHOW TABLES IN bronze;
SHOW TABLES IN bronze_audit;
SHOW TABLES IN bronze_dlq;

+---------+--------------+-----------+
|namespace|tableName     |isTemporary|
+---------+--------------+-----------+
|bronze   |order_payments|false      |
|bronze   |orders        |false      |
|bronze   |order_reviews |false      |
|bronze   |customers     |false      |
|bronze   |products      |false      |
|bronze   |order_items   |false      |
|bronze   |sellers       |false      |
+---------+--------------+-----------+

+------------+--------------------+-----------+
|namespace   |tableName           |isTemporary|
+------------+--------------------+-----------+
|bronze_audit|order_items_audit   |false      |
|bronze_audit|sellers_audit       |false      |
|bronze_audit|order_payments_audit|false      |
|bronze_audit|orders_audit        |false      |
|bronze_audit|order_reviews_audit |false      |
|bronze_audit|customers_audit     |false      |
|bronze_audit|products_audit      |false      |
+------------+--------------------+-----------+

+----------+------------------+----------

In [9]:
for item in TOPIC_SUFFIX:
    df = spark.table(f"{CATALOG}.{BRONZE_NAMESPACE}.{item}").limit(5)
    df.show()
    df = spark.table(f"{CATALOG}.{BRONZE_NAMESPACE}_audit.{item}_audit").limit(5)
    df.show()

+--------------------+----------------------+-----------+------------+----------------+----------------+----+-------------+------------+---------------+--------------------+--------+--------------------+
|           seller_id|seller_zip_code_prefix|seller_city|seller_state|      created_at|      updated_at|__op|      __ts_ms|kafka_offset|kafka_partition|     kafka_ingest_ts|batch_id|     spark_ingest_ts|
+--------------------+----------------------+-----------+------------+----------------+----------------+----+-------------+------------+---------------+--------------------+--------+--------------------+
|f235413f677777a07...|                 17250|     bariri|          SP|1766340942943182|1766340942943182|   r|1772975551021|           0|              0|2026-03-08 13:12:...|       0|2026-03-09 00:03:...|
|53e4c6e0f4312d4d2...|                 13920|   pedreira|          SP|1766340942943182|1766340942943182|   r|1772975551021|           1|              0|2026-03-08 13:12:...|       0|20

In [10]:
spark.catalog.clearCache()  # clears all cached tables
spark.stop()  